# CC-ResDiff on Colab: Global Color-Consistency Supervision for Residual-Space Diffusion SR

This notebook trains and evaluates **CC-ResDiff** (baseline SRDiff vs. SRDiff + the Global Color-Consistency loss) on a small CelebA subset, sized to fit a single 4GB GPU / Colab T4, per `CC-ResDiff_spec.md`.

Pipeline: mount Drive -> get code -> prepare a CelebA subset (16x16 -> 64x64, 4x) -> train Stage 1 RRDB -> train Stage 2 diffusion (baseline, then CC-ResDiff) -> evaluate both (PSNR/SSIM/LPIPS/FID/color-error).

Everything runs **inside your mounted Google Drive folder** so `data/` and `checkpoints/` persist across Colab disconnects -- re-running a training cell resumes from the last checkpoint automatically.

## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Mount Drive and get the code

The CC-ResDiff modifications (GCC loss, small-scale configs, color-error metric, this notebook) live in **this local `SRDiff/` folder**, not on the upstream `LeiaLi/SRDiff` GitHub repo. To use them in Colab, pick ONE:

- **Option A (recommended): push this folder to your own GitHub repo**, then set `GITHUB_REPO_URL` below.
- **Option B: upload this `SRDiff/` folder into your Google Drive** (e.g. `MyDrive/CC-ResDiff/SRDiff`) and leave `GITHUB_REPO_URL` empty -- the cell will use it directly from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- configure these two lines ----
GITHUB_REPO_URL = ''  # e.g. 'https://github.com/<you>/SRDiff-CC-ResDiff.git'; leave '' to use Drive copy
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/CC-ResDiff'
# ------------------------------------

import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
repo_dir = f'{DRIVE_PROJECT_DIR}/SRDiff'

if GITHUB_REPO_URL:
    if not os.path.exists(repo_dir):
        !git clone "{GITHUB_REPO_URL}" "{repo_dir}"
    else:
        print(f'{repo_dir} already exists, pulling latest changes')
        !cd "{repo_dir}" && git pull
else:
    assert os.path.exists(repo_dir), (
        f'{repo_dir} not found. Upload your modified SRDiff/ folder there, '
        'or set GITHUB_REPO_URL above.')

%cd {repo_dir}
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Prepare a CelebA subset

Downloads the aligned+cropped CelebA dataset via `torchvision` (identity/attribute labels are not needed, only images + the train/valid/test partition file), reorganizes it into the layout `data_gen/celeb_a.py` expects, then packs a **subsampled** binary dataset (`max_train_imgs`/`max_valid_imgs`/`max_test_imgs` in `configs/celeb_a_small.yaml`, default 3000/300/300) at 64x64 HR / 16x16 LR (4x).

**Known flakiness:** `torchvision`'s CelebA download pulls from a Google Drive link that is frequently quota-limited and may fail with a `RuntimeError`/`gdown` error. If it fails, download CelebA manually instead (e.g. the [Kaggle mirror](https://www.kaggle.com/datasets/jessicali9530/celeba-dataset)) and place it so you end up with `data/raw/CelebA/Img/img_align_celeba/*.jpg` and `data/raw/CelebA/Eval/list_eval_partition.txt`, matching the layout the symlink step below creates; then skip straight to the `data_gen/celeb_a.py` cell.

In [ ]:
import os

raw_root = f'{repo_dir}/data/raw/CelebA'
os.makedirs(raw_root, exist_ok=True)

# torchvision downloads to <root>/celeba/{img_align_celeba, list_eval_partition.txt, ...}
import torchvision
torchvision.datasets.CelebA(root=f'{repo_dir}/data/raw', split='all', download=True)

tv_dir = f'{repo_dir}/data/raw/celeba'
os.makedirs(f'{raw_root}/Img', exist_ok=True)
os.makedirs(f'{raw_root}/Eval', exist_ok=True)
if not os.path.exists(f'{raw_root}/Img/img_align_celeba'):
    os.symlink(f'{tv_dir}/img_align_celeba', f'{raw_root}/Img/img_align_celeba')
if not os.path.exists(f'{raw_root}/Eval/list_eval_partition.txt'):
    os.symlink(f'{tv_dir}/list_eval_partition.txt', f'{raw_root}/Eval/list_eval_partition.txt')
print('CelebA raw data ready at', raw_root)

In [ ]:
!python data_gen/celeb_a.py --config configs/celeb_a_small.yaml

## 4. Stage 1: train the RRDB conditioning net (small)

This is unmodified from upstream SRDiff -- CC-ResDiff only touches Stage 2. Re-running this cell after a disconnect resumes from the last checkpoint in `checkpoints/rrdb_celebA_small`.

In [ ]:
!python tasks/trainer.py --config configs/rrdb/celeb_a_pretrain_small.yaml \
    --exp_name rrdb_celebA_small

## 5. Stage 2: train the diffusion model -- baseline vs. CC-ResDiff

Two arms, same data/model/schedule/seed, differing only in `use_color_loss` (see `configs/diffsr_celeb_small.yaml` vs. `configs/diffsr_celeb_small_cc.yaml`). Run both cells (they can be run in either order); each resumes from its own checkpoint dir if interrupted.

In [ ]:
# Baseline (unmodified SRDiff)
!python tasks/trainer.py --config configs/diffsr_celeb_small.yaml \
    --exp_name diffsr_celebA_small_baseline \
    --hparams="rrdb_ckpt=checkpoints/rrdb_celebA_small"

In [ ]:
# Proposed: SRDiff + CC-ResDiff (GCC loss)
!python tasks/trainer.py --config configs/diffsr_celeb_small_cc.yaml \
    --exp_name diffsr_celebA_small_cc \
    --hparams="rrdb_ckpt=checkpoints/rrdb_celebA_small"

## 6. Evaluate both arms

`tasks/evaluate.py` runs inference over the test split, saves generated PNGs, and reports PSNR/SSIM/LPIPS/LR-PSNR/**color_error** (new CC-ResDiff metric) plus **FID**. Metrics are also written to `checkpoints/<exp_name>/results_*/metrics.json`.

In [ ]:
!python tasks/evaluate.py --config configs/diffsr_celeb_small.yaml \
    --exp_name diffsr_celebA_small_baseline

In [ ]:
!python tasks/evaluate.py --config configs/diffsr_celeb_small_cc.yaml \
    --exp_name diffsr_celebA_small_cc

In [ ]:
import glob
import json

import pandas as pd


def load_latest_metrics(exp_name):
    paths = sorted(glob.glob(f'checkpoints/{exp_name}/results_*/metrics.json'))
    assert paths, f'no metrics.json found for {exp_name}; run evaluate.py first'
    with open(paths[-1]) as f:
        return json.load(f)


rows = {
    'SRDiff (baseline)': load_latest_metrics('diffsr_celebA_small_baseline'),
    'SRDiff + CC-ResDiff': load_latest_metrics('diffsr_celebA_small_cc'),
}
df = pd.DataFrame(rows).T
df

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ['psnr', 'ssim', 'color_error', 'fid']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4 * len(metrics_to_plot), 4))
for ax, m in zip(axes, metrics_to_plot):
    if m in df.columns:
        df[m].plot(kind='bar', ax=ax, title=m)
        ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 7. (Optional) Ablations: `lambda_color` and `color_pool_size`

Per the spec's experimental plan: sweep `lambda_color` in `{0.01, 0.1, 0.5, 1.0}` and pooling `size` in `{4, 8, 16}`. Each run is a full Stage-2 training + eval, so this is compute-heavy -- uncomment and run only the settings you need, and consider lowering `max_updates` further for a quick scan before committing to full runs.

In [ ]:
lambda_color_values = [0.01, 0.1, 0.5, 1.0]
color_pool_sizes = [4, 8, 16]

# for lam in lambda_color_values:
#     exp_name = f'diffsr_celebA_small_cc_lam{lam}'
#     !python tasks/trainer.py --config configs/diffsr_celeb_small_cc.yaml \\
#         --exp_name {exp_name} \\
#         --hparams="rrdb_ckpt=checkpoints/rrdb_celebA_small,lambda_color={lam}"
#     !python tasks/evaluate.py --config configs/diffsr_celeb_small_cc.yaml --exp_name {exp_name}

# for size in color_pool_sizes:
#     exp_name = f'diffsr_celebA_small_cc_size{size}'
#     !python tasks/trainer.py --config configs/diffsr_celeb_small_cc.yaml \\
#         --exp_name {exp_name} \\
#         --hparams="rrdb_ckpt=checkpoints/rrdb_celebA_small,color_pool_size={size}"
#     !python tasks/evaluate.py --config configs/diffsr_celeb_small_cc.yaml --exp_name {exp_name}